<a href="https://colab.research.google.com/github/AliceFranca0/tcc-deteccao-fake-news-ptbr/blob/main/notebooks/Fase5_FineTuning_BERTimbau.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!nvidia-smi

Sun Sep 20 19:58:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/TCC"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!pip install transformers datasets accelerate evaluate scikit-learn -q

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(f"{BASE}/dados/fakerecogna_bruto.csv")
df = df.dropna(subset=["Noticia", "Classe"]).reset_index(drop=True)
df["Classe"] = df["Classe"].astype(int)

# MESMO random_state=42 dos baselines — essencial para a comparação ser justa
X_train, X_test, y_train, y_test = train_test_split(
    df["Noticia"], df["Classe"],
    test_size=0.2, random_state=42, stratify=df["Classe"]
)

print(f"Treino: {len(X_train)} | Teste: {len(X_test)}")

Treino: 9521 | Teste: 2381


In [11]:
from transformers import AutoTokenizer
import datasets

MODELO = "neuralmind/bert-base-portuguese-cased"

# do_lower_case=False é obrigatório no BERTimbau — o modelo diferencia
# maiúsculas de minúsculas, e isso carrega informação em português
tokenizer = AutoTokenizer.from_pretrained(MODELO, do_lower_case=False)

def tokenizar(lote):
    return tokenizer(
        lote["texto"],
        truncation=True,      # corta textos longos demais
        padding="max_length", # iguala o tamanho de todos
        max_length=256        # 256 tokens cobre a maioria das notícias do corpus
    )

ds_train = datasets.Dataset.from_dict({"texto": X_train.tolist(), "labels": y_train.tolist()})
ds_test  = datasets.Dataset.from_dict({"texto": X_test.tolist(),  "labels": y_test.tolist()})

ds_train = ds_train.map(tokenizar, batched=True)
ds_test  = ds_test.map(tokenizar, batched=True)

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/9521 [00:00<?, ? examples/s]

Map:   0%|          | 0/2381 [00:00<?, ? examples/s]

In [12]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

modelo = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=2)

def calcular_metricas(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precisao, revocacao, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "acuracia": accuracy_score(labels, preds),
        "f1": f1,
        "precisao": precisao,
        "revocacao": revocacao,
    }

args = TrainingArguments(
    output_dir="/content/drive/MyDrive/TCC/bertimbau_resultados",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,          # acelera o treino na GPU T4
    logging_steps=50,
    report_to="none",   # evita pedir login no Weights & Biases
)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    compute_metrics=calcular_metricas,
)

trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Acuracia,F1,Precisao,Revocacao
1,0.150283,0.109552,0.965981,0.965576,0.976784,0.954622
2,0.080182,0.146037,0.963461,0.962387,0.991095,0.935294
3,0.012644,0.118011,0.976480,0.976351,0.981324,0.971429


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1788, training_loss=0.09784725930966787, metrics={'train_runtime': 541.1919, 'train_samples_per_second': 52.778, 'train_steps_per_second': 3.304, 'total_flos': 3757620537123840.0, 'train_loss': 0.09784725930966787, 'epoch': 3.0})

In [18]:
resultados = trainer.evaluate()
print(resultados)

# Previsões para a matriz de confusão
previsoes = trainer.predict(ds_test)
y_pred_bert = previsoes.predictions.argmax(-1)

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred_bert, target_names=["Fake", "Real"]))
print(confusion_matrix(y_test, y_pred_bert))

# Salvar o modelo treinado (necessário para os Blocos B e C)
trainer.save_model(f"{BASE}/modelos/modelo_bertimbau_final")
tokenizer.save_pretrained(f"{BASE}/modelos/modelo_bertimbau_final")

Training Loss,Validation Loss,Epoch,Acuracia,F1,Precisao,Revocacao
0.012644,0.118011,3,0.976480,0.976351,0.981324,0.971429


{'eval_loss': 0.11801108717918396, 'eval_acuracia': 0.9764804703905922, 'eval_f1': 0.9763513513513513, 'eval_precisao': 0.9813242784380306, 'eval_revocacao': 0.9714285714285714}


              precision    recall  f1-score   support

        Fake       0.97      0.98      0.98      1191
        Real       0.98      0.97      0.98      1190

    accuracy                           0.98      2381
   macro avg       0.98      0.98      0.98      2381
weighted avg       0.98      0.98      0.98      2381

[[1169   22]
 [  34 1156]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/TCC/modelos/modelo_bertimbau_final/tokenizer_config.json',
 '/content/drive/MyDrive/TCC/modelos/modelo_bertimbau_final/tokenizer.json')

In [19]:
# Exportar previsões para a consolidação comparativa (Fase 7)
import pandas as pd

pd.DataFrame({
    "classe_real": y_test.values,
    "previsao_bertimbau": y_pred_bert,
}).to_csv(f"{BASE}/dados/previsoes_bertimbau.csv", index=False, encoding="utf-8-sig")

In [20]:
!pip install -q huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CAMINHO = "/content/drive/MyDrive/TCC/modelo_bertimbau_final"
tok = AutoTokenizer.from_pretrained(CAMINHO)
mod = AutoModelForSequenceClassification.from_pretrained(CAMINHO)

# Sem isso, o modelo devolve "LABEL_0" e "LABEL_1" para quem o usar
mod.config.id2label = {0: "FAKE", 1: "REAL"}
mod.config.label2id = {"FAKE": 0, "REAL": 1}

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [21]:
NOME_REPO = "bertimbau-fake-news-ptbr"

mod.push_to_hub(NOME_REPO)
tok.push_to_hub(NOME_REPO)

print("Enviado! Acesse em: https://huggingface.co/AliceFranca0/" + NOME_REPO)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._naz7nm/model.safetensors:   2%|1         | 7.95MB /  436MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Enviado! Acesse em: https://huggingface.co/AliceFranca0/bertimbau-fake-news-ptbr
